# Randomness and Stochasticity

This notebook introduces you to how you would inject randomness into a simulation module.

Broadly speaking, there are two categories of randomness:

    1. Time-dependent randomness
    2. State-dependent randomness

## Time-Dependent Randomness

Time dependent randomness is the easier case because the random trajectories are determined *a priori*. This means you can generate random trajectories with whatever method you want and then use them in your simulation. An example is shown in the CometMirror notebook, but repeated here for convenience.

First, let's load up an instance of `CometMirror` and use `popsim.stochastic` to generate random trajectories for all particle confinement time scalars.

In [ ]:
%load_ext autoreload
%autoreload 2

import jax

from popsim.simulate import MultiCases, SimInput, make_time_base, simulate
from popsim.simulators.comet_mirror.scenarios.sparc_prd import build_comet_mirror_config
from popsim.stochastic import generate_random_walks
from popsim.visualize import visualize_inputs

# Initialize the simulator.
model, state, inputs = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = make_time_base(t0=0.0, t1=5.0, dt=0.01)


# Diffusion mags specify the degree of randomness in the random walks.
diffusion_mags = {k: 0.25 for k in inputs.particle_confinement_scalar.keys()}
n_samps = 10

# Generate random walks and visualize them.
random_walks = generate_random_walks(
    jax.random.PRNGKey(42),  # Seed the random number generator.
    n_samps,
    time_base,
    inputs.particle_confinement_scalar,  # Specify a dictionary of inital conditions.
    diffusion_mags,  # Specify the degree of randomness in the random walks.
)
visualize_inputs(random_walks, time_base)

**Creating Simulation Cases**
We can see from below that `random_walks` is a list of interpolation objects that interpolate the dictionary of particle species. We can directly assign this to the `inputs` structure and use `MultiCases` to specify that the list is a list of different simulation cases.

We can then construct the `SimInput` object and call `generate_sim_cases()` to generate the simulation cases. We can then run the simulation like usual.

In [ ]:
from pprint import pprint

from popsim.visualize import visualize_time_series

pprint(random_walks)
inputs.particle_confinement_scalar = MultiCases(cases=random_walks)

sim_inputs = SimInput(time=time_base, initial_state=state, inputs=inputs)
sim_inputs = sim_inputs.generate_sim_cases()

# Run the simulation.
ds = simulate(model, sim_inputs)

visualize_vars = [
    "inputs.particle_confinement_scalar.FuelSpecies.Tritium",
    "inputs.particle_confinement_scalar.FuelSpecies.Deuterium",
    "inputs.particle_confinement_scalar.AtomicSpecies.Tungsten",
    "state.density_state.vol_avg_ion.FuelSpecies.Tritium",
    "state.density_state.vol_avg_ion.FuelSpecies.Deuterium",
    "state.density_state.vol_avg_ion.AtomicSpecies.Tungsten",
]
visualize_time_series(ds, visualize_vars)

## State-Dependent Randomness

However, there may be cases where randomness is state-dependent, or it is more convenient to generate randomness inside the module itself.

Let's consider a toy problem: the Lorenz system with measurement noise. Suppose, for whatever reason, that the measurement noise starts scaling when the distance from the origin exceeds a certain threshold. 


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt


def noise_variance(base_variance: float, distance_threshold: float, distance: float, distance_scale_factor: float):
    additional_noise = jnp.maximum(0.0, distance_scale_factor * (distance - distance_threshold))
    return base_variance + additional_noise


distances = jnp.linspace(0, 30, 100)
variances = noise_variance(0.01, 10, distances, 0.05)
plt.plot(distances, variances)

### Using the PRNGModule

In Jax-land, all random operations are deterministic, given a `PRNGKeyArray` as input. This is for a good reason: it ensures reproducibility and state encapsulation.

In fact, all random generators on computers are actually pseudo-random number generators (PRNGs); Jax just exposes this fact.

To achieve randomness in Jax land, you need to create a `PRNGKeyArray`, and then pass it to PRNG functions. The same key used twice will generate the same random numbers. To achieve different random numbers, you need to split the key. To learn more, we recommend reading the [Jax Documentation](https://jax.readthedocs.io/en/latest/random-numbers.html).
```python
# Generate some random numbers.
key = jax.random.key(0)
random_numbers = jax.random.normal(key, shape=(10,))

# This will generate the same random numbers as above as it uses the same key.
random_numbers2 = jax.random.normal(key, shape=(10,))

# This will generate a different set of random numbers.
key, subkey = jax.random.split(key)
random_numbers3 = jax.random.normal(subkey, shape=(10,))
```

To make this all a bit easier for you, there is a `PRNGModule` which randomly resets the seed at every time step. You can simply grab `PRNGModule.Output.key` to use as your key.

⚠️ **Remember to split your keys:** if you use the same key twice, you will get the same random numbers, so make sure you split the key after using it.


### Creating a LorenzWithNoise Module

Okay, let's create a module modelling the Lorenz system with state-dependent measurement noise using the pre-existing `BasicLorenz` and `PRNGModule` and simulate it. We recommend reviewing the [introduction to modules](./intro_to_modules.ipynb) if the module structure below seems confusing at all.

In [ ]:
import chex
import jax

from popsim import TimeDepModule
from popsim.modules.module_examples import BasicLorenz
from popsim.modules.prng import PRNGModule
from popsim.simulate import SimInput, make_time_base, simulate


class LorenzWithNoise(TimeDepModule):
    @chex.dataclass
    class Config:
        prng_module: PRNGModule
        lorenz_module: BasicLorenz

    @chex.dataclass
    class State:
        prng_state: PRNGModule.State
        lorenz_state: BasicLorenz.State

    @chex.dataclass
    class Inputs:
        lorenz_inputs: BasicLorenz.Inputs
        base_variance: float
        distance_threshold: float
        distance_scale_factor: float

    @chex.dataclass
    class Output:
        noisy_observation: BasicLorenz.State
        variance: float

    config: Config

    def __init__(self, config: Config):
        self.config = config

    def __call__(self, state: "State", inputs: "Inputs") -> tuple["State", "Output"]:
        # Call the PRNG and Lorenz modules.
        prng_state_out, prng_output = self.config.prng_module(state.prng_state, None)
        key = prng_output.key  # Grab the key from the PRNG output.
        lorenz_state_out, lorenz_output = self.config.lorenz_module(state.lorenz_state, inputs.lorenz_inputs)

        # Compute state-dependent noise variance.
        variance = noise_variance(
            inputs.base_variance,
            inputs.distance_threshold,
            lorenz_output.distance_from_origin,
            distance_scale_factor=inputs.distance_scale_factor,
        )

        # Sample noise for lorenz state.
        lorenz_noise = jax.random.normal(key, shape=(3,)) * variance

        out = LorenzWithNoise.Output(
            noisy_observation=BasicLorenz.State(
                x=state.lorenz_state.x + lorenz_noise[0],
                y=state.lorenz_state.y + lorenz_noise[1],
                z=state.lorenz_state.z + lorenz_noise[2],
            ),
            variance=variance,
        )

        state_out = LorenzWithNoise.State(prng_state=prng_state_out, lorenz_state=lorenz_state_out)
        return state_out, out


# Define the module, time_base, initial_state, and inputs.
config = LorenzWithNoise.Config(
    prng_module=PRNGModule(),
    lorenz_module=BasicLorenz(config=BasicLorenz.Config()),
)
module = LorenzWithNoise(config=config)

initial_state = LorenzWithNoise.State(
    prng_state=PRNGModule.State(seed=42),
    lorenz_state=BasicLorenz.State(x=1.0, y=1.0, z=1.0),
)

inputs = LorenzWithNoise.Inputs(
    lorenz_inputs=BasicLorenz.Inputs(sigma=10.0, rho=28.0, beta=8.0 / 3.0),
    base_variance=0.01,
    distance_threshold=10.0,
    distance_scale_factor=0.05,
)
time_base = make_time_base(t0=0.0, t1=3.0, dt=0.01)

ds = simulate(module=module, sim_inputs=SimInput(time=time_base, initial_state=initial_state, inputs=inputs))

### Visualizing the Measured State + Variance

Now we can visualize the noisy measurements of the Lorenz system and also the variance of noise as a function of time.

In [ ]:
# Visualize the results.
import holoviews as hv

(
    hv.Scatter3D((ds["output.noisy_observation.x"], ds["output.noisy_observation.y"], ds["output.noisy_observation.z"]))
    + ds["output.variance"].hvplot()
)